# Module 297 — Direct Prompt Injection Lab

This notebook accompanies HTB Section 4. It is a teaching and observation worksheet for the authorized Prompt Leak 1 and direct-order examples. It does not send network requests, connect to SMTP, or automate prompts. You interact with the HTB lab through its provided interface and record what you observe here.

Keep secrets from the lab out of this notebook and out of Git. Record outcomes such as ‘refused’ or ‘partial disclosure,’ not the secret itself.

## Lab connection: SSH as a set of pipes

Use the exact server address and SSH port shown after starting the HTB lab. The placeholders below are not valid values. The tunnel keeps your local web browser pointed at the lab's remote services:

```text
ssh htb-stdnt@<SERVER_IP> -p <PORT> -R 8000:127.0.0.1:8000 -L 2525:127.0.0.1:25 -L 5000:127.0.0.1:80 -N
```

- `-p <PORT>` selects the SSH service port.
- `-L 5000:127.0.0.1:80` makes your computer's port 5000 lead to the lab's web port 80; browse to `http://127.0.0.1:5000`.
- `-L 2525:127.0.0.1:25` similarly forwards your local port 2525 to the lab's SMTP port 25.
- `-R 8000:127.0.0.1:8000` lets the lab connect back through SSH to port 8000 on your computer.
- `-N` keeps the tunnel open without starting a shell or a remote command.

The HTB lesson says SSH is not configured for code execution. VPN connectivity gets you onto the Academy network; SSH forwarding carries only the listed connections. Stop the tunnel when finished.

## Prompt families in the lesson

A **direct prompt injection** puts attacker-controlled instructions in the user's own message. HTB's examples test whether the model can keep its intended rules in force when the user reframes the task. These are lab experiments, not guaranteed bypasses:

1. **Rule change or authority claim:** append a supposed exception and claim to qualify. Text saying ‘I am an admin’ does not authenticate a user.
2. **Storytelling/context switch:** ask for a poem, story, or dialogue involving the protected content. This tests whether the secrecy instruction carries into the new task.
3. **Translation or spell-checking:** frame earlier instructions as text to transform.
4. **Summary or repetition:** ask for a summary, the first sentence, or a particular word.
5. **Encoding:** request Base64, ROT13, or reversed text. A model can produce plausible-looking but invalid encodings; verify with real software.
6. **Indirect inference:** request hints or fragments. Several small disclosures may reveal the whole secret.

Responses can vary between attempts. Capture the exact prompt and a safe description of the outcome, then compare runs. Do not assume one refusal proves the secret is protected.

## A small, controlled test plan

Start with the baseline prompt, then change one strategy at a time. Repeat only as allowed by HTB's lab rules. Keep the target and the scope limited to the current Academy lab. For each run, record the strategy, your prompt, a redacted outcome, and whether it worked. Never paste the lab's secret into notebook output.

In [ ]:
# Manual test log. These records stay in this Python session unless you save the notebook.
# Do not put secret values, credentials, or tokens in the prompt or outcome fields.
observations = []

def record_observation(strategy, prompt_summary, outcome, repeat_number=1):
    """Store one redacted result from an authorized HTB lab interaction.

    outcome should be a short label such as 'refused', 'partial disclosure',
    'invalid response', or 'price changed'—never the secret itself.
    """
    allowed_outcomes = {
        'refused', 'partial disclosure', 'disclosed (redacted)',
        'invalid response', 'price changed', 'no visible change', 'other (redacted)'
    }
    if outcome not in allowed_outcomes:
        raise ValueError(f'Choose a redacted outcome from: {sorted(allowed_outcomes)}')

    observations.append({
        'strategy': strategy,
        'prompt_summary': prompt_summary,
        'outcome': outcome,
        'repeat_number': repeat_number,
    })
    return observations[-1]

# Example format only; it does not contact the HTB lab.
# record_observation('baseline', 'plain request for the protected value', 'refused')

In [ ]:
# Count redacted outcomes by strategy to compare repeated attempts.
# A count is a summary of this small lab sample, not a general success probability.
from collections import Counter, defaultdict

counts_by_strategy = defaultdict(Counter)
for observation in observations:
    counts_by_strategy[observation['strategy']][observation['outcome']] += 1

for strategy, counts in sorted(counts_by_strategy.items()):
    print(f'{strategy}: {dict(counts)}')

## The order example: integrity and trusted application logic

In HTB's drinks-order example, the model calculates a total. The lesson shows a prompt that changes an item's price and produces a cheaper result. This is an **integrity** problem: untrusted input influenced a value used for a business decision. A production server should load its own prices, authenticate the user, check discount eligibility, calculate the total, and validate any model output before acting. A model-generated claim of authority or price must not grant permission.

For your lab notes, record whether the response was accepted by the application and whether the calculated values matched the trusted menu. Do not use a real ordering or payment system for this exercise.

In [ ]:
# Review controls outside the model. Mark a control 'verified' only when
# the application behavior or trusted documentation provides evidence.
defense_review = {
    'Secrets are withheld unless the model needs them': 'not checked',
    'Authentication is checked by the application': 'not checked',
    'Prices and discounts come from trusted server data': 'not checked',
    'Model output is validated before consequential actions': 'not checked',
    'Tools use least privilege and action limits': 'not checked',
    'High-impact actions require appropriate confirmation': 'not checked',
}
defense_review